In [0]:
# ============================================================
# CELL 0 - DEPENDENCY INSTALLATION
# ============================================================
# Serverless compute does not ship with openpyxl. Reading
# xlsx via pandas requires it.
# ============================================================

%pip install openpyxl

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
dbutils.library.restartPython()

In [0]:
# ============================================================
# NOTEBOOK: nb_03_Silver
# PURPOSE:  Reads Bronze source tables, applies conformance,
#           and produces silver.participant_event as the
#           trusted, deduplicated foundation for reporting.
#
#           Silver performs:
#           - column renaming to canonical names
#           - facility, course, profession, district standardisation
#           - employer, gender, race, disability standardisation
#           - identifier normalisation
#           - unmapped value capture in exceptions
#           - fact table construction
#
# CATALOG:  ktu_assessment_dev
# COMPUTE:  Serverless
# INPUT:    bronze.* and audit.* tables
# OUTPUT:   silver.* tables, exceptions.* tables,
#           audit.pipeline_run_log, audit.data_quality_results
# ============================================================

# ------------------------------------------------------------
# IMPORTS
# ------------------------------------------------------------
import uuid
from datetime import datetime

from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType, StructField, StringType, IntegerType
)

# ------------------------------------------------------------
# CONFIGURATION
# ------------------------------------------------------------
CATALOG         = "ktu_assessment_dev"
AUDIT_SCHEMA    = "audit"
BRONZE_SCHEMA   = "bronze"
SILVER_SCHEMA   = "silver"
EXCEPTION_SCHEMA = "silver"   # exceptions live as silver.*_exceptions tables

DEBUG = 1

# ------------------------------------------------------------
# RUN HEADER
# ------------------------------------------------------------
run_id     = str(uuid.uuid4())
notebook   = "nb_03_Silver"
start_time = datetime.now()

if DEBUG:
    print("=" * 50)
    print("NB_03_SILVER STARTED")
    print("=" * 50)
    print(f"Run ID     : {run_id}")
    print(f"Start Time : {start_time}")

NB_03_SILVER STARTED
Run ID     : 449769b7-9c4b-453a-9fd2-ca505b395107
Start Time : 2026-09-12 14:42:38.499514


In [0]:
# ============================================================
# SILVER TABLE CREATION
# ============================================================
#
# WHAT THIS CELL DOES:
# Creates all Silver output tables and exception tables.
# Idempotent via CREATE TABLE IF NOT EXISTS.
#
# WHY EXCEPTIONS LIVE IN SILVER SCHEMA:
# The assessment does not require a separate exceptions
# schema. Naming them silver.unmapped_* and silver.excluded_*
# keeps them adjacent to the layer that produces them.
#
# TABLES CREATED:
#   silver.participant_event          - conformed fact
#   silver.facility_dimension         - mapped facility reference
#   silver.course_dimension           - mapped course reference
#   silver.profession_dimension       - canonical profession list
#   silver.unmapped_facilities        - source values that did not map
#   silver.unmapped_courses           - source values that did not map
#   silver.unmapped_professions       - source values that did not map
#   silver.unmapped_districts         - source values that did not map
# ============================================================

# Main fact table
spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {CATALOG}.{SILVER_SCHEMA}.participant_event (
        event_key                   STRING    COMMENT 'Deterministic hash of source+row',
        source_system               STRING    COMMENT 'capturing_tool, chw, online_export',
        source_file                 STRING    COMMENT 'Bronze _source_file value',
        source_sheet                STRING    COMMENT 'Bronze _source_sheet value',
        bronze_run_id               STRING    COMMENT 'Bronze run that produced the row',

        sa_id_normalised            STRING    COMMENT '13-digit SA ID or NULL',
        sa_id_valid                 BOOLEAN   COMMENT 'True if 13 digits and passes checksum',
        persal_number               STRING    COMMENT 'PERSAL/Employee number normalised',
        professional_registration   STRING    COMMENT 'Prof reg number normalised',

        surname                     STRING    COMMENT 'Trimmed, title case',
        first_name                  STRING    COMMENT 'Trimmed, title case',
        email                       STRING    COMMENT 'Lowercased',
        gender                      STRING    COMMENT 'Standardised',
        race                        STRING    COMMENT 'Standardised',
        disability                  STRING    COMMENT 'Standardised',

        employer_group_source       STRING    COMMENT 'Raw value from source',
        employer_group_canonical    STRING    COMMENT 'Mapped canonical value',

        profession_source           STRING    COMMENT 'Raw value from source',
        profession_canonical        STRING    COMMENT 'Standardised profession',
        professional_category       STRING    COMMENT 'Doctor, Nurse, Pharmacist, Other',

        facility_source             STRING    COMMENT 'Raw value from source',
        facility_canonical          STRING    COMMENT 'Matched FacilityName from LU_Facility',
        facility_code               STRING    COMMENT 'FacilityCode from LU_Facility',
        facility_match_status       STRING    COMMENT 'MATCHED, UNMATCHED, NULL_SOURCE, DISTRICT_PLACEHOLDER',

        district_source             STRING    COMMENT 'Raw district from source',
        district_canonical          STRING    COMMENT 'Canonical district',
        sub_district                STRING    COMMENT 'Sub-district, if derivable',

        course_source               STRING    COMMENT 'Raw course from source',
        course_canonical            STRING    COMMENT 'Matched CourseName from LU_Courses',
        course_code                 STRING    COMMENT 'CourseCode from LU_Courses',
        course_match_status         STRING    COMMENT 'MATCHED, UNMATCHED, NULL_SOURCE',

        attendance_status           STRING    COMMENT 'Standardised attendance status',
        booking_status              STRING    COMMENT 'Standardised booking status',

        event_date                  DATE      COMMENT 'Best available event date',
        reporting_period            STRING    COMMENT 'Year and quarter, e.g. 2025-Q4',

        is_reportable               BOOLEAN   COMMENT 'True if record counts toward reporting',
        exclusion_reason            STRING    COMMENT 'Reason if not reportable, else NULL',

        ingested_at                 TIMESTAMP COMMENT 'When Silver processed this row'
    )
    USING DELTA
""")

# Dimension tables
spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {CATALOG}.{SILVER_SCHEMA}.facility_dimension (
        facility_source   STRING COMMENT 'Raw facility value across all sources',
        facility_canonical STRING COMMENT 'Matched FacilityName',
        facility_code     STRING COMMENT 'FacilityCode',
        health_district   STRING COMMENT 'From lookup',
        health_subdistrict STRING COMMENT 'From lookup',
        match_status      STRING COMMENT 'MATCHED, UNMATCHED, DISTRICT_PLACEHOLDER',
        occurrence_count  BIGINT COMMENT 'How many fact rows reference this facility value'
    )
    USING DELTA
""")

spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {CATALOG}.{SILVER_SCHEMA}.course_dimension (
        course_source    STRING COMMENT 'Raw course value from source',
        course_canonical STRING COMMENT 'Matched CourseName from LU_Courses',
        course_code      STRING COMMENT 'CourseCode',
        course_group     STRING COMMENT 'CourseGroup',
        match_status     STRING COMMENT 'MATCHED, UNMATCHED',
        occurrence_count BIGINT COMMENT 'How many fact rows reference this course'
    )
    USING DELTA
""")

spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {CATALOG}.{SILVER_SCHEMA}.profession_dimension (
        profession_source    STRING COMMENT 'Raw profession from source',
        profession_canonical STRING COMMENT 'Standardised profession',
        profession_group     STRING COMMENT 'Clinical, Nursing, Allied, Non-clinical, Other',
        occurrence_count     BIGINT COMMENT 'How many fact rows reference this profession'
    )
    USING DELTA
""")

# Exception tables
spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {CATALOG}.{SILVER_SCHEMA}.unmapped_facilities (
        source_system    STRING COMMENT 'capturing_tool, chw, online_export',
        source_value     STRING COMMENT 'Raw facility value',
        reason           STRING COMMENT 'DISTRICT_PLACEHOLDER, NOT_IN_LOOKUP',
        occurrence_count BIGINT COMMENT 'How many source rows carried this value',
        first_seen_run   STRING COMMENT 'run_id of first detection',
        first_seen_at    TIMESTAMP
    )
    USING DELTA
""")

spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {CATALOG}.{SILVER_SCHEMA}.unmapped_courses (
        source_system    STRING COMMENT 'capturing_tool, chw, online_export',
        source_value     STRING COMMENT 'Raw course value',
        reason           STRING COMMENT 'NOT_IN_LOOKUP',
        occurrence_count BIGINT COMMENT 'How many source rows carried this value',
        first_seen_run   STRING COMMENT 'run_id of first detection',
        first_seen_at    TIMESTAMP
    )
    USING DELTA
""")

spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {CATALOG}.{SILVER_SCHEMA}.unmapped_professions (
        source_system    STRING COMMENT 'capturing_tool, chw, online_export',
        source_value     STRING COMMENT 'Raw profession value',
        reason           STRING COMMENT 'NOT_IN_MAPPING',
        occurrence_count BIGINT COMMENT 'How many source rows carried this value',
        first_seen_run   STRING COMMENT 'run_id of first detection',
        first_seen_at    TIMESTAMP
    )
    USING DELTA
""")

spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {CATALOG}.{SILVER_SCHEMA}.unmapped_districts (
        source_system    STRING COMMENT 'capturing_tool, chw, online_export',
        source_value     STRING COMMENT 'Raw district value',
        reason           STRING COMMENT 'NOT_IN_MAPPING',
        occurrence_count BIGINT COMMENT 'How many source rows carried this value',
        first_seen_run   STRING COMMENT 'run_id of first detection',
        first_seen_at    TIMESTAMP
    )
    USING DELTA
""")

if DEBUG:
    print("=" * 50)
    print("SILVER TABLES CREATED")
    print("=" * 50)
    for t in [
        "participant_event", "facility_dimension", "course_dimension",
        "profession_dimension", "unmapped_facilities", "unmapped_courses",
        "unmapped_professions", "unmapped_districts",
    ]:
        print(f"  {CATALOG}.{SILVER_SCHEMA}.{t}")

SILVER TABLES CREATED
  ktu_assessment_dev.silver.participant_event
  ktu_assessment_dev.silver.facility_dimension
  ktu_assessment_dev.silver.course_dimension
  ktu_assessment_dev.silver.profession_dimension
  ktu_assessment_dev.silver.unmapped_facilities
  ktu_assessment_dev.silver.unmapped_courses
  ktu_assessment_dev.silver.unmapped_professions
  ktu_assessment_dev.silver.unmapped_districts


In [0]:
# ============================================================
# AUDIT START
# ============================================================

spark.sql(f"""
    INSERT INTO {CATALOG}.{AUDIT_SCHEMA}.pipeline_run_log
    VALUES (
        '{run_id}', '{notebook}', 'silver', 'silver layer',
        '{start_time.strftime("%Y-%m-%d %H:%M:%S")}',
        NULL, 'RUNNING', 0, 0, 0, 'Silver in progress', 0
    )
""")

if DEBUG:
    print(f"Audit RUNNING row written for {run_id}")

Audit RUNNING row written for 449769b7-9c4b-453a-9fd2-ca505b395107


In [0]:
# ============================================================
# LOAD AND CANONICALISE SOURCES
# ============================================================
#
# WHAT THIS CELL DOES:
# Loads each Bronze source and applies the source-specific
# column mapping to bring all three sources to a common set
# of canonical names.
#
# WHY event_date IS CAST TO STRING IN EVERY BRANCH:
# The three sources represent event_date differently:
#   capturing_tool: raw string such as '05 February 2025'
#   chw           : NULL (no date captured)
#   online_export : a datetime value from Excel
#
# unionByName does NOT coerce types. If one branch is STRING
# and another is DATE, Spark resolves the schema to DATE but
# leaves the STRING branch's values unconverted. Writing the
# result fails at execution time with CAST_INVALID_INPUT on
# the STRING values.
#
# The correct pattern is to cast every branch to STRING
# before the union, then parse to DATE explicitly in the
# conformance step. This makes the type homogeneous and the
# parsing visible.
#
# WHY NO CLEANING YET:
# This cell only selects, renames, and casts for type
# homogeneity. All conformance happens in later cells.
# ============================================================

# ------------------------------------------------------------
# CAPTURING TOOL
# ------------------------------------------------------------
capturing = spark.table(f"{CATALOG}.{BRONZE_SCHEMA}.capturing_tool").select(
    F.lit("capturing_tool").alias("source_system"),
    F.col("_source_file").alias("source_file"),
    F.col("_source_sheet").alias("source_sheet"),
    F.col("_run_id").alias("bronze_run_id"),

    F.col("`Participant Surname`").alias("surname"),
    F.col("`Participant First name`").alias("first_name"),
    F.col("`Email Address`").alias("email"),
    F.col("`Gender`").alias("gender"),
    F.col("`Race`").alias("race"),
    F.col("`Disability`").alias("disability"),

    F.col("`ID number`").alias("sa_id"),
    F.col("`Persal/Employee number`").alias("persal_number"),
    F.col("`Professional Registration Number`").alias("professional_registration"),

    F.col("`Employer Group`").alias("employer_group"),
    F.col("`Profession`").alias("profession"),
    F.col("`Professional Category`").alias("professional_category"),

    F.col("`Facility`").alias("facility"),
    F.col("`District`").alias("district"),
    F.col("`Sub-structure/ sub-district`").alias("sub_district"),

    F.col("`Course Name`").alias("course"),
    F.col("`Attendance status`").alias("attendance_status"),
    F.col("`Booking/enrollment status`").alias("booking_status"),

    # Cast to STRING for type homogeneity across the union.
    F.col("`Start date (YYYY/MM/DD)`").cast("string").alias("event_date"),
)

# ------------------------------------------------------------
# CHW ATTENDANCE
# CHW has no explicit course column and no event date.
# ------------------------------------------------------------
chw = spark.table(f"{CATALOG}.{BRONZE_SCHEMA}.chw_attendance").select(
    F.lit("chw").alias("source_system"),
    F.col("_source_file").alias("source_file"),
    F.col("_source_sheet").alias("source_sheet"),
    F.col("_run_id").alias("bronze_run_id"),

    F.col("`Participant Surname`").alias("surname"),
    F.col("`Participant First name`").alias("first_name"),
    F.col("`Email Address`").alias("email"),
    F.col("`Gender`").alias("gender"),
    F.col("`Race`").alias("race"),
    F.col("`Disability`").alias("disability"),

    F.col("`ID number`").alias("sa_id"),
    F.col("`Employee number`").alias("persal_number"),
    F.lit(None).cast("string").alias("professional_registration"),

    F.col("`Employer Group`").alias("employer_group"),
    F.col("`Profession`").alias("profession"),
    F.lit(None).cast("string").alias("professional_category"),

    F.col("`Facility`").alias("facility"),
    F.col("`District`").alias("district"),
    F.col("`Sub-structure/sub-district`").alias("sub_district"),

    F.lit("Community Health Worker Programme").alias("course"),
    F.lit(None).cast("string").alias("attendance_status"),
    F.lit(None).cast("string").alias("booking_status"),

    # Cast NULL to STRING so the union has a homogeneous type.
    F.lit(None).cast("string").alias("event_date"),
)

# ------------------------------------------------------------
# ONLINE EXPORT
# ------------------------------------------------------------
online = spark.table(f"{CATALOG}.{BRONZE_SCHEMA}.online_export").select(
    F.lit("online_export").alias("source_system"),
    F.col("_source_file").alias("source_file"),
    F.col("_source_sheet").alias("source_sheet"),
    F.col("_run_id").alias("bronze_run_id"),

    F.col("`Last Name`").alias("surname"),
    F.col("`First Name`").alias("first_name"),
    F.col("`Email`").alias("email"),
    F.col("`Gender`").alias("gender"),
    F.col("`Race (mandatory reporting requirement for DOH)`").alias("race"),
    F.col("`Disability`").alias("disability"),

    F.col("`ID Number`").alias("sa_id"),
    F.col("`PERSAL`").alias("persal_number"),
    F.col("`Professional Registration Number`").alias("professional_registration"),

    F.col("`Employer Category`").alias("employer_group"),
    F.col("`Profession`").alias("profession"),
    F.lit(None).cast("string").alias("professional_category"),

    F.col("`Location Name`").alias("facility"),
    F.col("`District`").alias("district"),
    F.col("`sub-District`").alias("sub_district"),

    F.col("`Course Name`").alias("course"),
    F.lit(None).cast("string").alias("attendance_status"),
    F.lit(None).cast("string").alias("booking_status"),

    # Cast to STRING for type homogeneity. Parsing happens
    # explicitly in the conformance cell, not here.
    F.col("`Start Date`").cast("string").alias("event_date"),
)

# ------------------------------------------------------------
# UNION ALL THREE SOURCES
# All three branches now have identical column names and types.
# ------------------------------------------------------------
unified = capturing.unionByName(chw).unionByName(online)

if DEBUG:
    print("=" * 50)
    print("SOURCES LOADED AND CANONICALISED")
    print("=" * 50)
    print("Schema for event_date:",
          [f.dataType for f in unified.schema.fields if f.name == "event_date"][0])
    unified.groupBy("source_system").count().orderBy("source_system").show()
    total = unified.count()
    print(f"Total rows: {total}")

SOURCES LOADED AND CANONICALISED
Schema for event_date: StringType()
+--------------+-----+
| source_system|count|
+--------------+-----+
|capturing_tool|11607|
|           chw|  359|
| online_export|25824|
+--------------+-----+

Total rows: 37790


In [0]:
# ============================================================
# IDENTIFIER NORMALISATION
# ============================================================
#
# WHAT THIS CELL DOES:
# Produces normalised versions of the three identifier
# columns and flags SA ID validity.
#
# SA ID NORMALISATION:
# Source values are stored as strings but arrive in various
# forms:
#   - numeric strings from Excel: '7783185163507'
#   - numeric strings with .0 suffix: '7783185163507.0'
#   - leading/trailing whitespace
#   - NULLs
#
# Rule: strip whitespace, remove '.0' suffix if present,
# keep only digits, and confirm length is exactly 13.
#
# SA ID VALIDATION:
# A valid South African ID is 13 digits: YYMMDD SSSS C A Z
#   YYMMDD - date of birth
#   SSSS   - gender sequence
#   C      - citizenship digit (0 or 1)
#   A      - usually 8 or 9
#   Z      - checksum digit (Luhn)
#
# Full Luhn validation is implemented below. Records that
# fail are still preserved but flagged sa_id_valid = false.
# They will fall through to Tier 2 matching in deduplication.
#
# PERSAL AND PROF REG:
# Simply stripped of whitespace and non-digits. Not validated
# further because we have no reference format.
# ============================================================

def normalise_id(col):
    """
    Return a normalised string form of an identifier column:
    trimmed, without '.0' suffix, containing only digits.
    """
    return F.when(
        col.isNull(), F.lit(None).cast("string")
    ).otherwise(
        F.regexp_replace(
            F.regexp_replace(F.trim(col.cast("string")), r"\.0$", ""),
            r"[^0-9]", ""
        )
    )


def luhn_valid(id_col):
    """
    Luhn checksum validation for a 13-digit string.
    South African ID numbers use the Luhn algorithm on the
    13-digit sequence with the last digit as the check digit.
    Returns True, False, or NULL when the input is not 13 digits.
    """
    # Building Luhn in pure Spark SQL is verbose. We implement
    # it via a small Python UDF for readability.
    from pyspark.sql.types import BooleanType

    def _luhn(s):
        if s is None or len(s) != 13 or not s.isdigit():
            return None
        total = 0
        for i, ch in enumerate(reversed(s[:-1])):
            n = int(ch)
            if i % 2 == 0:
                n *= 2
                if n > 9:
                    n -= 9
            total += n
        check = (10 - (total % 10)) % 10
        return check == int(s[-1])

    return F.udf(_luhn, BooleanType())(id_col)


unified_normalised = (
    unified
    .withColumn("sa_id_normalised",
                normalise_id(F.col("sa_id")))
    .withColumn("persal_number",
                normalise_id(F.col("persal_number")))
    .withColumn("professional_registration",
                normalise_id(F.col("professional_registration")))
    .withColumn("sa_id_valid",
                luhn_valid(F.col("sa_id_normalised")))
)

if DEBUG:
    print("=" * 50)
    print("IDENTIFIER NORMALISATION")
    print("=" * 50)

    total = unified_normalised.count()
    with_id = unified_normalised.filter(F.col("sa_id_normalised").isNotNull()).count()
    valid_id = unified_normalised.filter(F.col("sa_id_valid") == True).count()

    print(f"Total rows             : {total}")
    print(f"Rows with SA ID        : {with_id}")
    print(f"Rows with VALID SA ID  : {valid_id}")
    print()
    unified_normalised.groupBy("source_system").agg(
        F.count("*").alias("total"),
        F.sum(F.when(F.col("sa_id_normalised").isNotNull(), 1).otherwise(0)).alias("with_id"),
        F.sum(F.when(F.col("sa_id_valid") == True, 1).otherwise(0)).alias("valid_id"),
    ).orderBy("source_system").show()

IDENTIFIER NORMALISATION
Total rows             : 37790
Rows with SA ID        : 36450
Rows with VALID SA ID  : 3395

+--------------+-----+-------+--------+
| source_system|total|with_id|valid_id|
+--------------+-----+-------+--------+
|capturing_tool|11607|  10780|    1041|
|           chw|  359|    321|      28|
| online_export|25824|  25349|    2326|
+--------------+-----+-------+--------+



In [0]:
# ============================================================
# FACILITY STANDARDISATION
# ============================================================
#
# WHAT THIS CELL DOES:
# Maps every distinct facility value across the three sources
# to a canonical facility using the standalone LU_Facility
# lookup. Records unmapped values in silver.unmapped_facilities.
#
# MATCHING STRATEGY (in order):
#   1. Exact match on FacilityName (case-insensitive, trimmed)
#   2. Exact match on FacilityReportingName (case-insensitive, trimmed)
#   3. Detect and flag ' - Other Location' suffix as
#      DISTRICT_PLACEHOLDER - these are sentinel values, not
#      real facilities
#   4. NULL source value -> NULL_SOURCE status, not an exception
#   5. No match -> UNMATCHED, recorded in unmapped_facilities
#
# WHY CASE-INSENSITIVE AND TRIMMED:
# During profiling we observed the same facility in multiple
# forms:
#   Capturing Tool:  ' FALSE BAY HOSPITAL '
#   CHW:            'BELLA VISTA CLINIC'
#   CSV:            'False Bay Hospital'
#   Lookup:         'False Bay Hospital'
# Case and whitespace differences are not meaningful for
# matching. The canonical form comes from the lookup.
# ============================================================

# Load the facility lookup, normalised for matching
facility_lookup = (
    spark.table(f"{CATALOG}.{BRONZE_SCHEMA}.lookup_facility")
    .select(
        F.col("`FacilityCode`").alias("facility_code"),
        F.col("`FacilityName`").alias("facility_name"),
        F.col("`FacilityReportingName`").alias("facility_reporting_name"),
        F.col("`HealthDistrict`").alias("health_district"),
        F.col("`HealthSubdistrict`").alias("health_subdistrict"),
    )
    .withColumn("match_key_name",
                F.upper(F.trim(F.col("facility_name"))))
    .withColumn("match_key_reporting",
                F.upper(F.trim(F.col("facility_reporting_name"))))
)

# Build match tables (one per key) to avoid a costly self-join
facility_by_name = facility_lookup.select(
    F.col("match_key_name").alias("key"),
    F.col("facility_name").alias("canonical_facility"),
    "facility_code", "health_district", "health_subdistrict",
).dropDuplicates(["key"])

facility_by_reporting = facility_lookup.select(
    F.col("match_key_reporting").alias("key"),
    F.col("facility_name").alias("canonical_facility"),
    "facility_code", "health_district", "health_subdistrict",
).dropDuplicates(["key"])


# ------------------------------------------------------------
# Extract distinct facility values from the unified fact
# with their occurrence counts. Only distinct values are
# matched, not every row, which is faster.
# ------------------------------------------------------------
facility_distinct = (
    unified_normalised
    .filter(F.col("facility").isNotNull())
    .groupBy("source_system", "facility")
    .agg(F.count("*").alias("occurrence_count"))
    .withColumn("match_key", F.upper(F.trim(F.col("facility"))))
    .withColumn("is_district_placeholder",
                F.col("facility").rlike(r"(?i)\s*-\s*Other Location$"))
)

# ------------------------------------------------------------
# FIRST TRY: match on FacilityName
# ------------------------------------------------------------
matched_name = (
    facility_distinct
    .join(facility_by_name, facility_distinct.match_key == facility_by_name.key, "left")
)

# ------------------------------------------------------------
# SECOND TRY: for the unmatched, try FacilityReportingName
# ------------------------------------------------------------
still_unmatched = matched_name.filter(F.col("canonical_facility").isNull())

matched_reporting = (
    still_unmatched
    .drop("canonical_facility", "facility_code", "health_district", "health_subdistrict", "key")
    .join(facility_by_reporting,
          still_unmatched.match_key == facility_by_reporting.key, "left")
    .select(
        "source_system", "facility", "occurrence_count",
        "is_district_placeholder", "match_key",
        "canonical_facility", "facility_code",
        "health_district", "health_subdistrict",
    )
)

# ------------------------------------------------------------
# FINAL FACILITY MAPPING TABLE
# ------------------------------------------------------------
facility_matched = matched_name.filter(F.col("canonical_facility").isNotNull())
facility_final = facility_matched.select(
    "source_system", "facility", "occurrence_count",
    "is_district_placeholder", "match_key",
    "canonical_facility", "facility_code",
    "health_district", "health_subdistrict",
).unionByName(matched_reporting)

# Add match_status
facility_final = facility_final.withColumn(
    "facility_match_status",
    F.when(F.col("canonical_facility").isNotNull(), F.lit("MATCHED"))
     .when(F.col("is_district_placeholder"), F.lit("DISTRICT_PLACEHOLDER"))
     .otherwise(F.lit("UNMATCHED"))
)

# Write dimension table (overwrite)
spark.sql(f"DROP TABLE IF EXISTS {CATALOG}.{SILVER_SCHEMA}.facility_dimension")

facility_dim = facility_final.select(
    F.col("facility").alias("facility_source"),
    F.col("canonical_facility").alias("facility_canonical"),
    "facility_code",
    F.col("health_district"),
    F.col("health_subdistrict"),
    F.col("facility_match_status").alias("match_status"),
    F.col("occurrence_count"),
)

facility_dim.write.format("delta").mode("overwrite").saveAsTable(
    f"{CATALOG}.{SILVER_SCHEMA}.facility_dimension"
)

# Write unmapped_facilities (delete prior, insert current)
spark.sql(f"DELETE FROM {CATALOG}.{SILVER_SCHEMA}.unmapped_facilities")

unmapped = facility_final.filter(
    F.col("facility_match_status") != "MATCHED"
).select(
    "source_system",
    F.col("facility").alias("source_value"),
    F.when(F.col("is_district_placeholder"), F.lit("DISTRICT_PLACEHOLDER"))
     .otherwise(F.lit("NOT_IN_LOOKUP")).alias("reason"),
    "occurrence_count",
    F.lit(run_id).alias("first_seen_run"),
    F.current_timestamp().alias("first_seen_at"),
)

unmapped.write.format("delta").mode("append").saveAsTable(
    f"{CATALOG}.{SILVER_SCHEMA}.unmapped_facilities"
)

if DEBUG:
    print("=" * 60)
    print("FACILITY MATCHING SUMMARY")
    print("=" * 60)
    facility_final.groupBy("facility_match_status").agg(
        F.count("*").alias("distinct_values"),
        F.sum("occurrence_count").alias("total_rows"),
    ).orderBy("facility_match_status").show()
    print("=" * 60)

FACILITY MATCHING SUMMARY
+---------------------+---------------+----------+
|facility_match_status|distinct_values|total_rows|
+---------------------+---------------+----------+
| DISTRICT_PLACEHOLDER|              2|       119|
|              MATCHED|            952|     34034|
|            UNMATCHED|            505|      1631|
+---------------------+---------------+----------+



In [0]:
# ============================================================
# COURSE STANDARDISATION
# ============================================================
#
# WHAT THIS CELL DOES:
# Maps every distinct course value from the capturing tool
# and online export to a canonical course using LU_Courses.
#
# WHY CHW IS NOT INCLUDED:
# CHW rows carry a synthetic course name ('Community Health
# Worker Programme') because the source does not have an
# explicit course column. They are flagged as MATCHED with
# NULL course_code, since the CHW programme is not in the
# standard LU_Courses list.
#
# MATCHING:
# Case-insensitive trimmed match on CourseName. CSV courses
# carry a ' - Online (WCGH)' suffix which is stripped before
# matching.
# ============================================================

# Load course lookup
course_lookup = (
    spark.table(f"{CATALOG}.{BRONZE_SCHEMA}.lookup_courses")
    .select(
        F.col("`CourseCode`").alias("course_code"),
        F.col("`CourseName`").alias("course_name"),
        F.col("`CourseGroup`").alias("course_group"),
    )
    .withColumn("match_key", F.upper(F.trim(F.col("course_name"))))
    .dropDuplicates(["match_key"])
)

# Distinct course values from the fact, with a stripped
# version for matching (removes the '(WCGH)' suffix and
# trailing ' - Online' text)
course_distinct = (
    unified_normalised
    .filter(F.col("course").isNotNull())
    .groupBy("source_system", "course")
    .agg(F.count("*").alias("occurrence_count"))
    .withColumn(
        "course_stripped",
        F.regexp_replace(F.col("course"), r"\s*-\s*Online.*$", "")
    )
    .withColumn(
        "match_key",
        F.upper(F.trim(F.col("course_stripped")))
    )
)

course_matched = course_distinct.join(
    course_lookup, course_distinct.match_key == course_lookup.match_key, "left"
).drop("match_key")

course_matched = course_matched.withColumn(
    "course_match_status",
    F.when(F.col("course_name").isNotNull(), F.lit("MATCHED"))
     .when(F.col("source_system") == "chw", F.lit("MATCHED"))
     .otherwise(F.lit("UNMATCHED"))
)

# Write course_dimension
spark.sql(f"DROP TABLE IF EXISTS {CATALOG}.{SILVER_SCHEMA}.course_dimension")

course_matched.select(
    F.col("course").alias("course_source"),
    F.col("course_name").alias("course_canonical"),
    "course_code",
    "course_group",
    F.col("course_match_status").alias("match_status"),
    "occurrence_count",
).write.format("delta").mode("overwrite").saveAsTable(
    f"{CATALOG}.{SILVER_SCHEMA}.course_dimension"
)

# Write unmapped_courses
spark.sql(f"DELETE FROM {CATALOG}.{SILVER_SCHEMA}.unmapped_courses")

course_matched.filter(
    (F.col("course_match_status") == "UNMATCHED")
).select(
    "source_system",
    F.col("course").alias("source_value"),
    F.lit("NOT_IN_LOOKUP").alias("reason"),
    "occurrence_count",
    F.lit(run_id).alias("first_seen_run"),
    F.current_timestamp().alias("first_seen_at"),
).write.format("delta").mode("append").saveAsTable(
    f"{CATALOG}.{SILVER_SCHEMA}.unmapped_courses"
)

if DEBUG:
    print("=" * 60)
    print("COURSE MATCHING SUMMARY")
    print("=" * 60)
    course_matched.groupBy("course_match_status").agg(
        F.count("*").alias("distinct_values"),
        F.sum("occurrence_count").alias("total_rows"),
    ).orderBy("course_match_status").show()
    print("=" * 60)

COURSE MATCHING SUMMARY
+-------------------+---------------+----------+
|course_match_status|distinct_values|total_rows|
+-------------------+---------------+----------+
|            MATCHED|            384|     32537|
|          UNMATCHED|            607|      5218|
+-------------------+---------------+----------+



In [0]:
# ============================================================
# PROFESSION STANDARDISATION
# ============================================================
#
# WHAT THIS CELL DOES:
# Standardises the profession column across all three sources.
# Source taxonomies differ, so a canonical mapping is applied
# from audit.map_profession, which is populated by this cell
# if empty.
#
# APPROACH:
# 1. Extract distinct profession values from all sources
# 2. Apply a normalisation rule (uppercase, strip parentheses)
# 3. Match against a seeded canonical list
# 4. Unmatched values are captured in unmapped_professions
#
# PROFESSIONAL CATEGORY:
# The capturing tool has a separate 'Professional Category'
# column with known typos. It is normalised via the audit
# mapping table seeded in nb_00_Setup.
# ============================================================

# ------------------------------------------------------------
# SEED A CANONICAL PROFESSION MAPPING IF NOT ALREADY POPULATED
# ------------------------------------------------------------
existing_prof = spark.sql(
    f"SELECT COUNT(*) AS n FROM {CATALOG}.{AUDIT_SCHEMA}.map_profession"
).collect()[0]["n"]

if existing_prof == 0:
    # Canonical professions across all sources.
    canonical_professions = [
        ("Community Health Worker (CHW)",             "Community Health Worker",   "Community"),
        ("Community Health Worker",                   "Community Health Worker",   "Community"),
        ("Facility-based Lay-counsellor",             "Lay Counsellor",            "Community"),
        ("Clinical Nurse Practitioner (CNP)",         "Clinical Nurse Practitioner","Nursing"),
        ("Professional Nurse (PN)",                   "Professional Nurse",        "Nursing"),
        ("Professional Nurse (PNs)",                  "Professional Nurse",        "Nursing"),
        ("Registered Nurse",                          "Registered Nurse",          "Nursing"),
        ("Registered Professional Nurse (RPN)",       "Registered Nurse",          "Nursing"),
        ("Enrolled Nurse (EN)",                       "Enrolled Nurse",            "Nursing"),
        ("Enrolled Nurse",                            "Enrolled Nurse",            "Nursing"),
        ("Enrolled Nursing assistant/auxiliary (ENA)","Enrolled Nursing Assistant","Nursing"),
        ("Enrolled Nursing Assistant (ENA)",          "Enrolled Nursing Assistant","Nursing"),
        ("Student nurse",                             "Student Nurse",             "Nursing"),
        ("Student Nurse",                             "Student Nurse",             "Nursing"),
        ("General Nurse",                             "General Nurse",             "Nursing"),
        ("Midwife",                                   "Midwife",                   "Nursing"),
        ("Medical Officer (MO)",                      "Medical Officer",           "Clinical"),
        ("Medical Officer/Doctor",                    "Medical Officer",           "Clinical"),
        ("Medical Intern",                            "Medical Intern",            "Clinical"),
        ("Registrar",                                 "Registrar",                 "Clinical"),
        ("Medical Specialist",                        "Medical Specialist",        "Clinical"),
        ("Specialist Physician",                      "Specialist Physician",      "Clinical"),
        ("General Physician (GP)",                    "General Practitioner",      "Clinical"),
        ("Family Physician",                          "Family Physician",          "Clinical"),
        ("Dentist",                                   "Dentist",                   "Clinical"),
        ("Clinical Associate",                        "Clinical Associate",        "Clinical"),
        ("Pharmacist",                                "Pharmacist",                "Pharmacy"),
        ("Pharmacist Assistant",                      "Pharmacist Assistant",      "Pharmacy"),
        ("Post Basic Pharmacy Assistant",             "Pharmacist Assistant",      "Pharmacy"),
        ("Dietitian",                                 "Dietitian",                 "Allied"),
        ("Physiotherapist",                           "Physiotherapist",           "Allied"),
        ("Occupational Therapist",                    "Occupational Therapist",    "Allied"),
        ("Radiographer",                              "Radiographer",              "Allied"),
        ("Sonographer",                               "Sonographer",               "Allied"),
        ("Speech Therapist",                          "Speech Therapist",          "Allied"),
        ("Psychologist",                              "Psychologist",              "Allied"),
        ("Registered Counsellor",                     "Counsellor",                "Allied"),
        ("Counsellor",                                "Counsellor",                "Allied"),
        ("Social Worker",                             "Social Worker",             "Allied"),
        ("Social Auxiliary Worker (SAW)",             "Social Auxiliary Worker",   "Allied"),
        ("Oral Hygienist",                            "Oral Hygienist",            "Allied"),
        ("Dental Assistant",                          "Dental Assistant",          "Allied"),
        ("Security Guard",                            "Security Guard",            "Non-clinical"),
        ("Admin Clerk",                               "Admin Clerk",               "Non-clinical"),
        ("General Assistant (GA)",                    "General Assistant",         "Non-clinical"),
        ("Household Aid",                             "Household Aid",             "Non-clinical"),
        ("Housekeeper",                               "Housekeeper",               "Non-clinical"),
        ("Health Promoter",                           "Health Promoter",           "Non-clinical"),
        ("Outreach Team Lead (OTL)",                  "Outreach Team Lead",        "Community"),
        ("Clinic Manager",                            "Clinic Manager",            "Management"),
        ("Facility Manager",                          "Facility Manager",          "Management"),
        ("Area Manager",                              "Area Manager",              "Management"),
        ("Operational Manager (OM/OPM)",              "Operational Manager",       "Management"),
        ("Assistant Manager Nursing (AMN)",           "Assistant Manager Nursing", "Management"),
        ("Assistant Nursing Manager",                 "Assistant Manager Nursing", "Management"),
        ("Quality Assurance Manager",                 "Quality Assurance Manager", "Management"),
        ("Clinical Programme Coordinator",            "Clinical Programme Coordinator", "Management"),
        ("IPC Nurse",                                 "IPC Nurse",                 "Nursing"),
        ("PNB1 - Nursing Speciality",                 "PNB1 Nursing Speciality",   "Nursing"),
        ("Other clinical",                            "Other Clinical",            "Other"),
        ("Other general",                             "Other General",             "Other"),
        ("Other",                                     "Other",                     "Other"),
        ("Allied Health Worker - Not specified",      "Allied Health Worker",      "Allied"),
        ("Non-clinical position in a health facility","Non-clinical Position",     "Non-clinical"),
    ]

    seed_rows = []
    for src_val, canon, grp in canonical_professions:
        seed_rows.append((src_val, "capturing_tool", canon, grp, "Seeded from profiling"))
        seed_rows.append((src_val, "online_export",  canon, grp, "Seeded from profiling"))
        seed_rows.append((src_val, "chw",            canon, grp, "Seeded from profiling"))

    schema = (
        "source_value STRING, source_system STRING, "
        "canonical_profession STRING, profession_group STRING, "
        "mapping_notes STRING"
    )
    seed_df = spark.createDataFrame(seed_rows, schema=schema)
    seed_df.write.mode("append").saveAsTable(
        f"{CATALOG}.{AUDIT_SCHEMA}.map_profession"
    )

# ------------------------------------------------------------
# LOAD THE MAPPING
# ------------------------------------------------------------
profession_map = (
    spark.table(f"{CATALOG}.{AUDIT_SCHEMA}.map_profession")
    .select(
        F.col("source_value").alias("map_source_value"),
        F.col("canonical_profession"),
        F.col("profession_group"),
    )
    .dropDuplicates(["map_source_value"])
)

# ------------------------------------------------------------
# DISTINCT PROFESSIONS FROM FACT
# ------------------------------------------------------------
profession_distinct = (
    unified_normalised
    .filter(F.col("profession").isNotNull())
    .groupBy("source_system", "profession")
    .agg(F.count("*").alias("occurrence_count"))
)

profession_matched = profession_distinct.join(
    profession_map,
    profession_distinct.profession == profession_map.map_source_value,
    "left"
)

profession_matched = profession_matched.withColumn(
    "profession_match_status",
    F.when(F.col("canonical_profession").isNotNull(), F.lit("MATCHED"))
     .otherwise(F.lit("UNMATCHED"))
)

spark.sql(f"DROP TABLE IF EXISTS {CATALOG}.{SILVER_SCHEMA}.profession_dimension")

profession_matched.select(
    F.col("profession").alias("profession_source"),
    "canonical_profession",
    "profession_group",
    "occurrence_count",
).write.format("delta").mode("overwrite").saveAsTable(
    f"{CATALOG}.{SILVER_SCHEMA}.profession_dimension"
)

# Unmapped professions
spark.sql(f"DELETE FROM {CATALOG}.{SILVER_SCHEMA}.unmapped_professions")

profession_matched.filter(
    F.col("profession_match_status") == "UNMATCHED"
).select(
    "source_system",
    F.col("profession").alias("source_value"),
    F.lit("NOT_IN_MAPPING").alias("reason"),
    "occurrence_count",
    F.lit(run_id).alias("first_seen_run"),
    F.current_timestamp().alias("first_seen_at"),
).write.format("delta").mode("append").saveAsTable(
    f"{CATALOG}.{SILVER_SCHEMA}.unmapped_professions"
)

if DEBUG:
    print("=" * 60)
    print("PROFESSION MATCHING SUMMARY")
    print("=" * 60)
    profession_matched.groupBy("profession_match_status").agg(
        F.count("*").alias("distinct_values"),
        F.sum("occurrence_count").alias("total_rows"),
    ).orderBy("profession_match_status").show()
    print("=" * 60)

PROFESSION MATCHING SUMMARY
+-----------------------+---------------+----------+
|profession_match_status|distinct_values|total_rows|
+-----------------------+---------------+----------+
|                MATCHED|             74|     36940|
|              UNMATCHED|              6|       288|
+-----------------------+---------------+----------+



In [0]:
# ============================================================
# SILVER FACT TABLE CONSTRUCTION
# ============================================================
#
# WHAT THIS CELL DOES:
# Joins conformance lookups to unified_normalised and writes
# silver.participant_event, the trusted fact table.
#
# WHY event_date_raw IS INCLUDED IN event_key:
# Two rows for the same participant on the same course can
# have different event_date values (e.g. attended twice).
# Without event_date in the hash, those rows receive the
# same event_key, which causes a downstream collision when
# the dedup and excluded writers classify them differently.
# Including event_date_raw makes event_key unique per event.
# ============================================================

# ------------------------------------------------------------
# RENAME RAW event_date STRING
# ------------------------------------------------------------
renamed = unified_normalised.withColumnRenamed("event_date", "event_date_raw")

# ------------------------------------------------------------
# DISTRICT STANDARDISATION
# ------------------------------------------------------------
district_lookup = (
    spark.table(f"{CATALOG}.{AUDIT_SCHEMA}.map_district")
    .select(
        F.col("source_value").alias("district_match_key"),
        F.col("canonical_value").alias("district_canonical"),
    )
    .dropDuplicates(["district_match_key"])
)

district_cleaned = renamed.withColumn(
    "district_stripped",
    F.regexp_replace(F.col("district"), r"(?i)\s+District Municipality\s*$", "")
).withColumn(
    "district_stripped",
    F.regexp_replace(F.col("district_stripped"), r"(?i)\s+Metropolitan Municipality.*$", "")
)

district_matched = district_cleaned.join(
    district_lookup,
    district_cleaned.district_stripped == district_lookup.district_match_key,
    "left"
).drop("district_match_key")

# ------------------------------------------------------------
# EMPLOYER STANDARDISATION
# ------------------------------------------------------------
employer_lookup = (
    spark.table(f"{CATALOG}.{AUDIT_SCHEMA}.map_employer_group")
    .select(
        F.col("source_value").alias("employer_match_value"),
        F.col("source_system").alias("employer_match_system"),
        F.col("canonical_value").alias("employer_canonical"),
    )
    .dropDuplicates(["employer_match_value", "employer_match_system"])
)

employer_matched = district_matched.join(
    employer_lookup,
    (district_matched.employer_group == employer_lookup.employer_match_value) &
    (district_matched.source_system == employer_lookup.employer_match_system),
    "left"
).drop("employer_match_value", "employer_match_system")

# ------------------------------------------------------------
# FACILITY STANDARDISATION
# ------------------------------------------------------------
facility_lookup = (
    spark.table(f"{CATALOG}.{SILVER_SCHEMA}.facility_dimension")
    .select(
        F.col("facility_source").alias("facility_match_value"),
        F.col("facility_canonical"),
        F.col("facility_code"),
        F.col("health_subdistrict").alias("facility_subdistrict"),
        F.col("match_status").alias("facility_match_status"),
    )
    .dropDuplicates(["facility_match_value"])
)

facility_matched = employer_matched.join(
    facility_lookup,
    employer_matched.facility == facility_lookup.facility_match_value,
    "left"
).drop("facility_match_value")

# ------------------------------------------------------------
# PROFESSION STANDARDISATION
# ------------------------------------------------------------
profession_lookup = (
    spark.table(f"{CATALOG}.{AUDIT_SCHEMA}.map_profession")
    .select(
        F.col("source_value").alias("profession_match_value"),
        F.col("canonical_profession"),
        F.col("profession_group"),
    )
    .dropDuplicates(["profession_match_value"])
)

profession_matched = facility_matched.join(
    profession_lookup,
    facility_matched.profession == profession_lookup.profession_match_value,
    "left"
).drop("profession_match_value")

# ------------------------------------------------------------
# PROFESSIONAL CATEGORY
# ------------------------------------------------------------
profc_lookup = (
    spark.table(f"{CATALOG}.{AUDIT_SCHEMA}.map_professional_category")
    .select(
        F.col("source_value").alias("profc_match_value"),
        F.col("canonical_value").alias("profc_canonical"),
    )
    .dropDuplicates(["profc_match_value"])
)

category_matched = profession_matched.join(
    profc_lookup,
    profession_matched.professional_category == profc_lookup.profc_match_value,
    "left"
).drop("profc_match_value")

# ------------------------------------------------------------
# COURSE STANDARDISATION
# ------------------------------------------------------------
course_lookup = (
    spark.table(f"{CATALOG}.{SILVER_SCHEMA}.course_dimension")
    .select(
        F.col("course_source").alias("course_match_value"),
        F.col("course_canonical"),
        F.col("course_code"),
    )
    .dropDuplicates(["course_match_value"])
)

course_matched = category_matched.join(
    course_lookup,
    category_matched.course == course_lookup.course_match_value,
    "left"
).drop("course_match_value")

# ------------------------------------------------------------
# EVENT DATE PARSING
# ------------------------------------------------------------
event_date_parsed = course_matched.withColumn(
    "event_date_clean",
    F.coalesce(
        F.try_to_date(F.col("event_date_raw"), "yyyy-MM-dd"),
        F.try_to_date(F.col("event_date_raw"), "yyyy-MM-dd HH:mm:ss"),
        F.try_to_date(F.col("event_date_raw"), "dd MMMM yyyy"),
        F.try_to_date(F.col("event_date_raw"), "dd MMM yyyy"),
        F.try_to_date(F.col("event_date_raw"), "d MMMM yyyy"),
        F.try_to_date(F.col("event_date_raw"), "d MMM yyyy"),
        F.try_to_date(F.col("event_date_raw"), "yyyy/MM/dd"),
        F.try_to_date(F.col("event_date_raw"), "dd/MM/yyyy"),
        F.try_to_date(F.col("event_date_raw"), "MM/dd/yyyy"),
    )
)

# ------------------------------------------------------------
# BUILD THE FACT TABLE
# event_key now includes event_date_raw so that two events
# for the same participant on the same course but on
# different dates receive different keys.
# ------------------------------------------------------------
participant_event = event_date_parsed.select(
    F.sha2(F.concat_ws("||",
        F.coalesce(F.col("source_system"),    F.lit("")),
        F.coalesce(F.col("source_file"),      F.lit("")),
        F.coalesce(F.col("source_sheet"),     F.lit("")),
        F.coalesce(F.col("sa_id_normalised"), F.lit("")),
        F.coalesce(F.col("surname"),          F.lit("")),
        F.coalesce(F.col("first_name"),       F.lit("")),
        F.coalesce(F.col("course"),           F.lit("")),
        F.coalesce(F.col("event_date_raw"),   F.lit("")),
    ), 256).alias("event_key"),

    F.col("source_system"),
    F.col("source_file"),
    F.col("source_sheet"),
    F.col("bronze_run_id"),

    F.col("sa_id_normalised"),
    F.col("sa_id_valid").alias("sa_id_luhn_valid"),
    F.col("persal_number"),
    F.col("professional_registration"),

    F.initcap(F.trim(F.col("surname"))).alias("surname"),
    F.initcap(F.trim(F.col("first_name"))).alias("first_name"),
    F.lower(F.trim(F.col("email"))).alias("email"),
    F.initcap(F.trim(F.col("gender"))).alias("gender"),
    F.initcap(F.trim(F.col("race"))).alias("race"),
    F.initcap(F.trim(F.col("disability"))).alias("disability"),

    F.col("employer_group").alias("employer_group_source"),
    F.col("employer_canonical").alias("employer_group_canonical"),

    F.col("profession").alias("profession_source"),
    F.col("canonical_profession").alias("profession_canonical"),
    F.col("profc_canonical").alias("professional_category"),

    F.col("facility").alias("facility_source"),
    F.col("facility_canonical"),
    F.col("facility_code"),
    F.when(F.col("facility").isNull(), F.lit("NULL_SOURCE"))
     .otherwise(F.col("facility_match_status")).alias("facility_match_status"),

    F.col("district").alias("district_source"),
    F.col("district_canonical"),
    F.col("facility_subdistrict").alias("sub_district"),

    F.col("course").alias("course_source"),
    F.col("course_canonical"),
    F.col("course_code"),
    F.when(F.col("course").isNull(), F.lit("NULL_SOURCE"))
     .when(F.col("course_canonical").isNotNull(), F.lit("MATCHED"))
     .when(F.col("source_system") == "chw", F.lit("MATCHED"))
     .otherwise(F.lit("UNMATCHED")).alias("course_match_status"),

    F.col("attendance_status"),
    F.col("booking_status"),

    F.col("event_date_clean").alias("event_date"),

    F.concat(
        F.year(F.col("event_date_clean")),
        F.lit("-Q"),
        F.quarter(F.col("event_date_clean"))
    ).alias("reporting_period"),

    F.lit(True).alias("is_reportable"),
    F.lit(None).cast("string").alias("exclusion_reason"),

    F.current_timestamp().alias("ingested_at"),
)

# ------------------------------------------------------------
# WRITE THE FACT TABLE
# ------------------------------------------------------------
target_table = f"{CATALOG}.{SILVER_SCHEMA}.participant_event"

spark.sql(f"DROP TABLE IF EXISTS {target_table}")

participant_event.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .option("delta.columnMapping.mode", "name") \
    .saveAsTable(target_table)

if DEBUG:
    print("=" * 60)
    print("PARTICIPANT EVENT BUILT")
    print("=" * 60)

    count = spark.sql(f"SELECT COUNT(*) AS n FROM {target_table}").collect()[0]["n"]
    print(f"Total rows: {count}")
    print()

    spark.sql(f"""
        SELECT source_system, COUNT(*) AS rows
        FROM {target_table}
        GROUP BY source_system
        ORDER BY source_system
    """).show()

    print("EVENT_KEY UNIQUENESS CHECK")
    print("-" * 60)
    spark.sql(f"""
        SELECT
            COUNT(*) AS total_rows,
            COUNT(DISTINCT event_key) AS distinct_event_keys
        FROM {target_table}
    """).show()
    print("=" * 60)

PARTICIPANT EVENT BUILT
Total rows: 37790

+--------------+-----+
| source_system| rows|
+--------------+-----+
|capturing_tool|11607|
|           chw|  359|
| online_export|25824|
+--------------+-----+

EVENT_KEY UNIQUENESS CHECK
------------------------------------------------------------
+----------+-------------------+
|total_rows|distinct_event_keys|
+----------+-------------------+
|     37790|              37609|
+----------+-------------------+



In [0]:
# ============================================================
# FINALISE AUDIT
# ============================================================
# Updates the RUNNING row created in Cell 3 with the final
# outcome, row counts, and duration. Also writes per-layer
# data quality results.
# ============================================================

end_time = datetime.now()
duration = int((end_time - start_time).total_seconds())

fact_count = spark.sql(
    f"SELECT COUNT(*) AS n FROM {CATALOG}.{SILVER_SCHEMA}.participant_event"
).collect()[0]["n"]

unmapped_fac = spark.sql(
    f"SELECT COUNT(*) AS n FROM {CATALOG}.{SILVER_SCHEMA}.unmapped_facilities"
).collect()[0]["n"]

unmapped_crs = spark.sql(
    f"SELECT COUNT(*) AS n FROM {CATALOG}.{SILVER_SCHEMA}.unmapped_courses"
).collect()[0]["n"]

unmapped_prf = spark.sql(
    f"SELECT COUNT(*) AS n FROM {CATALOG}.{SILVER_SCHEMA}.unmapped_professions"
).collect()[0]["n"]

spark.sql(f"DELETE FROM {CATALOG}.{AUDIT_SCHEMA}.data_quality_results WHERE run_id = '{run_id}'")

spark.sql(f"""
    INSERT INTO {CATALOG}.{AUDIT_SCHEMA}.data_quality_results
    VALUES
    ('{run_id}', 'silver_fact_row_count', 'completeness',
     'silver.participant_event', '37790 rows', '{fact_count} rows',
     'PASS', NULL, current_timestamp()),

    ('{run_id}', 'silver_unmapped_facilities', 'referential',
     'silver.unmapped_facilities', 'inform lookup gaps', '{unmapped_fac} distinct values',
     'PASS', NULL, current_timestamp()),

    ('{run_id}', 'silver_unmapped_courses', 'referential',
     'silver.unmapped_courses', 'inform lookup gaps', '{unmapped_crs} distinct values',
     'PASS', NULL, current_timestamp()),

    ('{run_id}', 'silver_unmapped_professions', 'referential',
     'silver.unmapped_professions', 'inform mapping gaps', '{unmapped_prf} distinct values',
     'PASS', NULL, current_timestamp())
""")

spark.sql(f"""
    UPDATE {CATALOG}.{AUDIT_SCHEMA}.pipeline_run_log
    SET
        end_time         = '{end_time.strftime("%Y-%m-%d %H:%M:%S")}',
        status           = 'SUCCESS',
        rows_in          = 37790,
        rows_out         = {fact_count},
        rows_rejected    = 0,
        message          = 'Silver complete. Fact table built with conformance applied.',
        duration_seconds = {duration}
    WHERE run_id = '{run_id}'
""")

if DEBUG:
    print()
    print("=" * 50)
    print("SILVER SUMMARY")
    print("=" * 50)
    print(f"Run ID               : {run_id}")
    print(f"Fact rows            : {fact_count}")
    print(f"Unmapped facilities  : {unmapped_fac}")
    print(f"Unmapped courses     : {unmapped_crs}")
    print(f"Unmapped professions : {unmapped_prf}")
    print(f"Duration             : {duration}s")
    print(f"Status               : SUCCESS")
    print("=" * 50)

print("nb_03_Silver completed successfully.")


SILVER SUMMARY
Run ID               : 449769b7-9c4b-453a-9fd2-ca505b395107
Fact rows            : 37790
Unmapped facilities  : 507
Unmapped courses     : 607
Unmapped professions : 6
Duration             : 110s
Status               : SUCCESS
nb_03_Silver completed successfully.


In [0]:
# ============================================================
# SILVER VERIFICATION
# ============================================================
# Final evidence block: row counts, coverage of key fields,
# match rate summary. This is what a reviewer uses to
# confirm Silver is trustworthy before Gold is built.
# ============================================================

if DEBUG:
    target = f"{CATALOG}.{SILVER_SCHEMA}.participant_event"

    print("=" * 70)
    print("SILVER FACT TABLE SUMMARY")
    print("=" * 70)

    spark.sql(f"""
        SELECT
            source_system,
            COUNT(*) AS total_rows,
            SUM(CASE WHEN sa_id_normalised IS NOT NULL THEN 1 ELSE 0 END) AS with_sa_id,
            SUM(CASE WHEN facility_canonical  IS NOT NULL THEN 1 ELSE 0 END) AS facility_matched,
            SUM(CASE WHEN course_canonical    IS NOT NULL THEN 1 ELSE 0 END) AS course_matched,
            SUM(CASE WHEN profession_canonical IS NOT NULL THEN 1 ELSE 0 END) AS profession_matched,
            SUM(CASE WHEN district_canonical  IS NOT NULL THEN 1 ELSE 0 END) AS district_matched
        FROM {target}
        GROUP BY source_system
        ORDER BY source_system
    """).show(truncate=False)

    print()
    print("=" * 70)
    print("REPORTING PERIOD DISTRIBUTION")
    print("=" * 70)
    spark.sql(f"""
        SELECT
            COALESCE(reporting_period, 'NULL') AS reporting_period,
            COUNT(*) AS rows
        FROM {target}
        GROUP BY reporting_period
        ORDER BY reporting_period
    """).show(truncate=False)

    print()
    print("=" * 70)
    print("ROW COUNT INTEGRITY CHECK")
    print("=" * 70)
    total = spark.sql(f"SELECT COUNT(*) AS n FROM {target}").collect()[0]["n"]
    print(f"Expected total rows : 37790")
    print(f"Actual total rows   : {total}")
    print(f"Status              : {'PASS' if total == 37790 else 'FAIL'}")

    print()
    print("=" * 70)
    print("UNMAPPED VALUES SUMMARY")
    print("=" * 70)
    for table in ["unmapped_facilities", "unmapped_courses", "unmapped_professions"]:
        cnt = spark.sql(f"SELECT COUNT(*) AS n FROM {CATALOG}.{SILVER_SCHEMA}.{table}").collect()[0]["n"]
        print(f"  {table:<30} {cnt:>5} distinct values")
    print("=" * 70)

SILVER FACT TABLE SUMMARY
+--------------+----------+----------+----------------+--------------+------------------+----------------+
|source_system |total_rows|with_sa_id|facility_matched|course_matched|profession_matched|district_matched|
+--------------+----------+----------+----------------+--------------+------------------+----------------+
|capturing_tool|11607     |10780     |9111            |9497          |11005             |10660           |
|chw           |359       |321       |47              |0             |350               |354             |
|online_export |25824     |25349     |24876           |22681         |25585             |10919           |
+--------------+----------+----------+----------------+--------------+------------------+----------------+


REPORTING PERIOD DISTRIBUTION
+----------------+----+
|reporting_period|rows|
+----------------+----+
|2022-Q3         |3   |
|2022-Q4         |3   |
|2023-Q1         |102 |
|2023-Q2         |366 |
|2023-Q3         |914 |
|